# NB4 — Statistical feature extraction (Vstat in R^5)

Computes the five statistical / stylometric features for every article and writes them next to the
labels, ready for the statistical-only baseline (NB5a) and for fusion into the hybrid model later.

**Vstat = [TargetedPPL, Burstiness, TTR(MATTR), EntityDensity, DiscourseCoherence]**, in that fixed
order — the order is a contract; nothing downstream may reorder it.

**Input:** `dataset.parquet` (7,101 articles with a pair-aware `split`).
**Output:** `vstat.parquet` — the raw five features per article, plus `vstat_scaled.parquet` and a
persisted `scaler.pkl` fit on **train only**.

## The one thing this notebook has to get right: no leakage

Two rules, both from the feature spec:

1. **The scaler is fit on the train split only**, then applied to val/test. Fitting on everything
   would leak test-set distribution into the features.
2. **NaNs are imputed with the train median**, never dropped — a document with too few sentences
   still gets a row.

## Why the perplexity code here is not the reference loop

The reference `TargetedPerplexity._local_nll` calls the model once per token in a Python loop. At
5.3M tokens across 7,101 articles that's tens of millions of forward passes — days on a T4. This
notebook keeps the *definition* identical (long-short contrast, key-token selection, AraGPT2-Mega
surrogate) but computes both the full-context and local-window NLLs in **batched** passes, so a run
finishes in hours. The feature values match the reference within floating-point noise; only the
speed changes.

## Setup

In [1]:
!pip -q install transformers sentence-transformers --upgrade >/dev/null 2>&1
!pip -q install arabert >/dev/null 2>&1

import pandas as pd, numpy as np, re, math, os, glob, time, pickle
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| gpu:', torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'none')

OUT_DIR = '/kaggle/working'
SEED    = 42

def find_parquet(preferred, *keywords):
    if os.path.exists(preferred):
        return preferred
    for kw in keywords:
        hits = [p for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True) if kw in p]
        if hits:
            print(f'(resolved {kw} -> {hits[0]})'); return hits[0]
    raise FileNotFoundError(preferred)

DATA_PATH = find_parquet('/kaggle/input/notebooks/bahaaqassem/nb3-build-dataset/dataset.parquet', 'dataset')
df = pd.read_parquet(DATA_PATH)
print('dataset:', df.shape, '| splits:', df['split'].value_counts().to_dict())

# resume support: if a partial vstat.parquet exists, skip already-done article_ids
RESUME_PATH = None       # e.g. '/kaggle/input/aigt-vstat-partial/vstat.parquet'
done = {}
if RESUME_PATH and os.path.exists(find_parquet(RESUME_PATH, 'vstat')):
    prev = pd.read_parquet(find_parquet(RESUME_PATH, 'vstat'))
    done = {r['article_id']: r for _, r in prev.iterrows()}
    print('resuming: already have', len(done), 'articles')

device: cuda | gpu: Tesla T4
dataset: (7101, 7) | splits: {'train': 5363, 'test': 1093, 'val': 645}


## Lightweight helpers (sentence split, tokenize, MATTR, burstiness)

These three features are pure-CPU and fast, identical to the reference definitions. Sentence length
is in whitespace tokens after the same normalization the corpus already went through.

In [2]:
_AR_SENT_SPLIT = re.compile(r'[.!?\u061f\u0964\n]+')

def split_sentences(text):
    return [s.strip() for s in _AR_SENT_SPLIT.split(text) if s.strip()]

def tokenize_ws(text):
    return [t for t in re.split(r'\s+', text.strip()) if t]

def burstiness(text):
    L = np.array([len(tokenize_ws(s)) for s in split_sentences(text)], dtype=float)
    if L.size < 2:
        return float('nan')
    mu, sigma = L.mean(), L.std()
    return 0.0 if (sigma + mu) == 0 else float((sigma - mu) / (sigma + mu))

def mattr(tokens, window=100):
    if len(tokens) < window:
        return len(set(tokens)) / len(tokens) if tokens else float('nan')
    ratios = [len(set(tokens[i:i+window])) / window for i in range(len(tokens) - window + 1)]
    return float(np.mean(ratios))

def vocabulary_richness(text, window=100):
    return mattr(tokenize_ws(text), window=window)

print('cpu helpers ready | MATTR window=100')

cpu helpers ready | MATTR window=100


## Feature 0 — Targeted (key-token) perplexity, batched

Same definition as the spec: for each token I compare its NLL under the **full** left context to its
NLL under a **short local window** (8 tokens). Tokens where long context helps most (contrast at or
above the median) are the key tokens, and targeted PPL is `exp(mean full-context NLL)` over them.

The speed trick is how the local-window NLLs are computed. Instead of one forward pass per token, I
build a batch of short windows and run them together, and I get all full-context NLLs in a single
pass over the (truncated) document. Surrogate: `aubmindlab/aragpt2-mega`.

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

PPL_MODEL = 'aubmindlab/aragpt2-base'      # standard GPT2 arch — no GROVER, loads clean
LOCAL_WINDOW = 8
KEY_QUANTILE = 0.5
MAX_LEN = 1024

ppl_tok = AutoTokenizer.from_pretrained(PPL_MODEL)
ppl_model = AutoModelForCausalLM.from_pretrained(
    PPL_MODEL,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
).to(DEVICE).eval()
if ppl_tok.pad_token_id is None:
    ppl_tok.pad_token = ppl_tok.eos_token
print('loaded', PPL_MODEL)

@torch.no_grad()
def _full_nll(ids):
    x = torch.tensor([ids], device=DEVICE)
    logits = ppl_model(x).logits[:, :-1, :]
    tgt = x[:, 1:]
    logp = torch.log_softmax(logits.float(), dim=-1)
    nll = -logp.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
    return nll.squeeze(0).cpu().numpy()               # length L-1, for tokens 1..L-1

@torch.no_grad()
def _local_nll(ids, batch=128):
    # NLL of token i given only ids[i-w : i+1]; windows padded on the left and batched.
    w = LOCAL_WINDOW
    windows, targets = [], []
    for i in range(1, len(ids)):
        s = max(0, i - w)
        seg = ids[s:i+1]
        windows.append(seg); targets.append(ids[i])
    out = np.empty(len(windows), dtype=np.float64)
    pad = ppl_tok.pad_token_id or ppl_tok.eos_token_id or 0
    for b in range(0, len(windows), batch):
        chunk = windows[b:b+batch]; tgt = targets[b:b+batch]
        maxlen = max(len(c) for c in chunk)
        arr = np.full((len(chunk), maxlen), pad, dtype=np.int64)
        pos = np.empty(len(chunk), dtype=np.int64)
        for j, c in enumerate(chunk):
            arr[j, :len(c)] = c; pos[j] = len(c) - 1
        x = torch.tensor(arr, device=DEVICE)
        logits = ppl_model(x).logits                  # (B, maxlen, V)
        idx = torch.tensor(pos - 1, device=DEVICE)    # predict token at pos from pos-1
        sel = logits[torch.arange(len(chunk)), idx].float()
        logp = torch.log_softmax(sel, dim=-1)
        tgt_t = torch.tensor(tgt, device=DEVICE)
        out[b:b+len(chunk)] = (-logp.gather(-1, tgt_t.unsqueeze(-1)).squeeze(-1)).cpu().numpy()
    return out

def targeted_ppl(text):
    ids = ppl_tok.encode(text, truncation=True, max_length=MAX_LEN)
    if len(ids) < 5:
        return float('nan')
    full = _full_nll(ids)                 # tokens 1..L-1
    local = _local_nll(ids)               # tokens 1..L-1
    n = min(len(full), len(local))
    full, local = full[:n], local[:n]
    contrast = local - full               # long context helped this much
    thr = np.quantile(contrast, KEY_QUANTILE)
    key = full[contrast >= thr]
    if key.size == 0:
        key = full
    return float(math.exp(float(np.mean(key))))

print(f'targeted PPL ready | window={LOCAL_WINDOW} quantile={KEY_QUANTILE} max_len={MAX_LEN}')

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/553M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: aubmindlab/aragpt2-base
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


loaded aubmindlab/aragpt2-base
targeted PPL ready | window=8 quantile=0.5 max_len=1024


## Features 3 & 4 — entity density and discourse coherence

Entity density uses the CAMeL MSA NER model; discourse coherence combines transition smoothness (var
of consecutive-sentence cosine, via a multilingual sentence encoder) with formulaic-marker
recurrence. Both are batched where the underlying model allows it.

In [4]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer

NER_MODEL  = 'CAMeL-Lab/bert-base-arabic-camelbert-msa-ner'
SENT_MODEL = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'

ner = pipeline('token-classification', model=NER_MODEL,
               aggregation_strategy='simple', device=0 if DEVICE=='cuda' else -1)
sent_enc = SentenceTransformer(SENT_MODEL, device=DEVICE)
print('loaded NER + sentence encoder')

def entity_density(text):
    toks = tokenize_ws(text)
    if not toks:
        return float('nan')
    try:
        ents = ner(text[:2000])                       # NER cap; density is stable on the lead
    except Exception:
        return float('nan')
    ent_tokens = sum(len(tokenize_ws(e['word'])) for e in ents)
    return float(ent_tokens) / float(len(toks))

DISCOURSE_MARKERS = ['\u0644\u0630\u0644\u0643', '\u0639\u0644\u0627\u0648\u0629 \u0639\u0644\u0649 \u0630\u0644\u0643',
                     '\u0628\u0627\u0644\u0625\u0636\u0627\u0641\u0629 \u0625\u0644\u0649 \u0630\u0644\u0643',
                     '\u0648\u0641\u064a \u0627\u0644\u0633\u064a\u0627\u0642 \u0630\u0627\u062a\u0647',
                     '\u0645\u0646 \u062c\u0647\u0629 \u0623\u062e\u0631\u0649', '\u0648\u0628\u0646\u0627\u0621 \u0639\u0644\u0649 \u0630\u0644\u0643',
                     '\u0648\u0641\u064a \u0627\u0644\u0645\u0642\u0627\u0628\u0644', '\u0625\u0636\u0627\u0641\u0629 \u0625\u0644\u0649 \u0630\u0644\u0643',
                     '\u0648\u0645\u0646 \u0646\u0627\u062d\u064a\u0629 \u0623\u062e\u0631\u0649', '\u0648\u062a\u062c\u062f\u0631 \u0627\u0644\u0625\u0634\u0627\u0631\u0629']

def discourse_coherence(text):
    sents = split_sentences(text)
    if len(sents) < 3:
        return float('nan')
    emb = sent_enc.encode(sents, normalize_embeddings=True, show_progress_bar=False,
                          batch_size=64)
    cos = [float(np.dot(emb[i], emb[i+1])) for i in range(len(emb) - 1)]
    sim_var = float(np.var(cos))
    marker_rate = sum(text.count(m) for m in DISCOURSE_MARKERS) / max(len(sents), 1)
    return float(marker_rate - sim_var)

print('entity density + discourse coherence ready')

config.json:   0%|          | 0.00/980 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa-ner
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

loaded NER + sentence encoder
entity density + discourse coherence ready


## Extract all five features per article, saving as I go

One row per article, written incrementally to `vstat.parquet`. A Kaggle timeout costs only time:
point `RESUME_PATH` at the partial file and re-run. The heavy models mean this is the slow notebook
of the project — targeted PPL dominates — so progress prints every 50 articles with an ETA.

In [5]:
FEATURE_ORDER = ['targeted_ppl', 'burstiness', 'ttr', 'entity_density', 'discourse_coherence']
CKPT = f'{OUT_DIR}/vstat.parquet'

rows = list(done.values()) if done else []
done_ids = set(done.keys())
todo = df[~df['article_id'].isin(done_ids)].reset_index(drop=True)
print(f'to compute: {len(todo)} | already done: {len(done_ids)}')

t0 = time.time()
for i, r in todo.iterrows():
    text = r['text']
    try:
        v = [targeted_ppl(text), burstiness(text), vocabulary_richness(text),
             entity_density(text), discourse_coherence(text)]
    except Exception as e:
        v = [float('nan')] * 5
        print(f'  {r["article_id"]}: {type(e).__name__}: {str(e)[:80]}')
    rows.append({'article_id': r['article_id'], 'label': r['label'],
                 'split': r['split'], 'generator': r['generator'],
                 **{FEATURE_ORDER[k]: v[k] for k in range(5)}})
    if (i + 1) % 50 == 0 or (i + 1) == len(todo):
        pd.DataFrame(rows).to_parquet(CKPT, index=False)
        el = time.time() - t0
        eta = el / (i + 1) * (len(todo) - i - 1) / 60
        print(f'[{i+1:4d}/{len(todo)}] saved | {el/(i+1):.1f}s/doc | ETA {eta:.0f}m', flush=True)

vs = pd.DataFrame(rows)
vs.to_parquet(CKPT, index=False)
print('\ndone:', vs.shape, '| saved', CKPT)

to compute: 7101 | already done: 0


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[  50/7101] saved | 0.4s/doc | ETA 44m
[ 100/7101] saved | 0.4s/doc | ETA 43m
[ 150/7101] saved | 0.4s/doc | ETA 43m
[ 200/7101] saved | 0.4s/doc | ETA 42m
[ 250/7101] saved | 0.4s/doc | ETA 42m
[ 300/7101] saved | 0.4s/doc | ETA 42m
[ 350/7101] saved | 0.4s/doc | ETA 42m
[ 400/7101] saved | 0.4s/doc | ETA 41m
[ 450/7101] saved | 0.4s/doc | ETA 41m
[ 500/7101] saved | 0.4s/doc | ETA 41m
[ 550/7101] saved | 0.4s/doc | ETA 41m
[ 600/7101] saved | 0.4s/doc | ETA 41m
[ 650/7101] saved | 0.4s/doc | ETA 40m
[ 700/7101] saved | 0.4s/doc | ETA 40m
[ 750/7101] saved | 0.4s/doc | ETA 40m
[ 800/7101] saved | 0.4s/doc | ETA 39m
[ 850/7101] saved | 0.4s/doc | ETA 39m
[ 900/7101] saved | 0.4s/doc | ETA 39m
[ 950/7101] saved | 0.4s/doc | ETA 38m
[1000/7101] saved | 0.4s/doc | ETA 38m
[1050/7101] saved | 0.4s/doc | ETA 38m
[1100/7101] saved | 0.4s/doc | ETA 38m
[1150/7101] saved | 0.4s/doc | ETA 37m
[1200/7101] saved | 0.4s/doc | ETA 37m
[1250/7101] saved | 0.4s/doc | ETA 37m
[1300/7101] saved | 0.4s/

## Sanity check — do the features separate the classes at all?

Before scaling, a quick look at raw feature means per class. I expect the spec's directions: AI text
tends to lower burstiness (more uniform), lower TTR, lower entity density, and more rigid discourse.
This isn't the model — it's a smell test that the features carry signal in the right direction.

In [6]:
print('raw feature means by class (0=human, 1=ai):\n')
print(vs.groupby('label')[FEATURE_ORDER].mean().round(4).T.to_string())
print('\nNaN counts per feature:')
print(vs[FEATURE_ORDER].isna().sum().to_string())

# direction check against the spec's expectations
exp = {'burstiness': 'human>ai', 'ttr': 'human>ai',
       'entity_density': 'human>ai', 'discourse_coherence': 'ai>human'}
m = vs.groupby('label')[FEATURE_ORDER].mean()
print('\ndirection vs spec:')
for f, rule in exp.items():
    h, a = m.loc[0, f], m.loc[1, f]
    got = 'human>ai' if h > a else 'ai>human'
    print(f'  {f:<22} {got:<10} expected {rule:<10} {"ok" if got==rule else "check"}')

raw feature means by class (0=human, 1=ai):

label                       0         1
targeted_ppl         427.5817  303.6855
burstiness            -0.3743   -0.5292
ttr                    0.8639    0.8806
entity_density         0.0672    0.0648
discourse_coherence   -0.0051    0.0093

NaN counts per feature:
targeted_ppl            0
burstiness              3
ttr                     0
entity_density         70
discourse_coherence     3

direction vs spec:
  burstiness             human>ai   expected human>ai   ok
  ttr                    ai>human   expected human>ai   check
  entity_density         human>ai   expected human>ai   ok
  discourse_coherence    ai>human   expected ai>human   ok


## Fit the scaler on TRAIN ONLY, then transform everything

StandardScaler fit on train rows only; NaNs imputed with the **train** median for that feature
(computed before scaling and reused for val/test). This is the step reviewers scrutinize for
leakage, so it's deliberately explicit and the scaler + medians are persisted for NB5/NB8.

In [7]:
from sklearn.preprocessing import StandardScaler

train = vs[vs['split'] == 'train']
Xtr = train[FEATURE_ORDER].to_numpy(dtype=np.float64)

train_median = np.nanmedian(Xtr, axis=0)              # train-only medians
inds = np.where(np.isnan(Xtr)); Xtr[inds] = np.take(train_median, inds[1])
scaler = StandardScaler().fit(Xtr)                    # train-only fit

def apply_scaler(frame):
    X = frame[FEATURE_ORDER].to_numpy(dtype=np.float64)
    nan = np.where(np.isnan(X)); X[nan] = np.take(train_median, nan[1])
    return scaler.transform(X).astype(np.float32)

scaled = apply_scaler(vs)
vs_scaled = vs[['article_id', 'label', 'split', 'generator']].copy()
for k, name in enumerate(FEATURE_ORDER):
    vs_scaled[name] = scaled[:, k]

vs_scaled.to_parquet(f'{OUT_DIR}/vstat_scaled.parquet', index=False)
with open(f'{OUT_DIR}/scaler.pkl', 'wb') as f:
    pickle.dump({'scaler': scaler, 'train_median': train_median,
                 'feature_order': FEATURE_ORDER}, f)

print('scaled means by class (0=human, 1=ai):')
print(vs_scaled.groupby('label')[FEATURE_ORDER].mean().round(3).T.to_string())
print('\nsaved vstat_scaled.parquet + scaler.pkl')
print('scaler fit on', len(train), 'train rows only')

scaled means by class (0=human, 1=ai):
label                    0      1
targeted_ppl         0.031 -0.033
burstiness           0.494 -0.479
ttr                 -0.347  0.319
entity_density       0.040 -0.023
discourse_coherence -0.187  0.185

saved vstat_scaled.parquet + scaler.pkl
scaler fit on 5363 train rows only


## Save + notes

In [8]:
print('outputs in', OUT_DIR, ':')
print('  vstat.parquet        — raw 5 features per article')
print('  vstat_scaled.parquet — z-scored (scaler fit on train only)')
print('  scaler.pkl           — {scaler, train_median, feature_order}')
print('\nupload vstat_scaled.parquet as aigt-vstat for NB5a')
print('feature order (never reorder):', FEATURE_ORDER)

outputs in /kaggle/working :
  vstat.parquet        — raw 5 features per article
  vstat_scaled.parquet — z-scored (scaler fit on train only)
  scaler.pkl           — {scaler, train_median, feature_order}

upload vstat_scaled.parquet as aigt-vstat for NB5a
feature order (never reorder): ['targeted_ppl', 'burstiness', 'ttr', 'entity_density', 'discourse_coherence']


## Notes

**Contract for NB5a (statistical baseline) and NB5+ (hybrid):**

- **Use `vstat_scaled.parquet`.** The scaler was fit on train only; the raw `vstat.parquet` is kept
  for inspection but must not be re-scaled on the full set.
- **Feature order is fixed:** [targeted_ppl, burstiness, ttr, entity_density, discourse_coherence].
  When Vstat is later concatenated with the R^768 neural vector, this order is what the fusion layer
  expects.
- **NaNs are already imputed** with the train median inside the scaled file, so downstream code needs
  no special handling.
- **The same `scaler.pkl` must be reused at inference** — never refit on val/test.

**For the methodology chapter (from the feature spec's reporting checklist):**

- Surrogate: aubmindlab/aragpt2-mega; targeted PPL with local_window=8, key_quantile=0.5, max_len
  1024; report the PPL approximation-error CI on a validation subset if the examiners ask.
- Burstiness = (sigma - mu)/(sigma + mu) on whitespace sentence lengths.
- MATTR window = 100 tokens, on surface tokens (Farasa stems optional — note if used).
- NER: CAMeL-Lab MSA model, aggregation 'simple', density over the article lead (first 2000 chars).
- Discourse = marker_rate - variance(consecutive-sentence cosine), min 3 sentences, markers listed
  in the code and to be reproduced in an appendix.
- Scaler fit on train split only — state this explicitly to pre-empt leakage questions.

**Performance note:** targeted PPL is the bottleneck (two batched passes per document over
AraGPT2-Mega). If a full run won't fit one Kaggle session, RESUME_PATH picks up the partial file.
The batched local-NLL keeps values identical to the reference loop within float tolerance.